# Synthetic Rank-One Test

In [197]:
import numpy as np
from ssl_shareability_metric.ssl_encoder_shareability import ssl_shareability
from ssl_shareability_metric.whiten_and_center import whiten_and_center
import torch
from torch.utils.data import TensorDataset, DataLoader
from ssl_shareability_metric.shared_encoder import SharedEncoder
from ssl_shareability_metric.seperate_encoder import SeperateEncoder
import copy

torch.manual_seed(42)

## Create rank 1 future representations for synthetic low mid and high sharability

In [198]:
rng = np.random.default_rng(42)

current = rng.normal(size=(5000, 13))

u_low = np.zeros(13)
v_low = np.zeros(13)

u_low[0] = 1.0
v_low[1] = 1.0

M_low = 0.9 * np.outer(u_low, v_low)

future_low_raw = current @ M_low.T

u_mid = np.zeros(13)
v_mid = np.zeros(13)

u_mid[0] = 1.0
v_mid[0] = 0.5
v_mid[1] = np.sqrt(0.75)

M_mid = 0.9 * np.outer(u_mid, v_mid)

future_mid_raw = current @ M_mid.T

u_high = np.zeros(13)
v_high = np.zeros (13)

u_high[0] = 1
v_high[0] = 1

M_high = 0.9 * np.outer(u_high, v_high)

future_high_raw = current @ M_high.T

noise_cov = np.eye(13) - M_low @ M_low.T

noise_eigenvalues, noise_eigenvectors = np.linalg.eigh(noise_cov)

noise_eigenvalues_sqrt = np.sqrt(noise_eigenvalues)

noise_sqrt = noise_eigenvectors @ np.diag(noise_eigenvalues_sqrt) @ noise_eigenvectors.T

raw_noise = rng.normal(size=current.shape)

noise = raw_noise @ noise_sqrt

future_low = future_low_raw + noise
future_mid = future_mid_raw + noise
future_high = future_high_raw + noise

print(f"current: {current.shape}")
print(f"future low: {future_low.shape}")
print(f"future mid: {future_mid.shape}")
print(f"future high: {future_high.shape}")

current: (5000, 13)
future low: (5000, 13)
future mid: (5000, 13)
future high: (5000, 13)


## Create splits

In [199]:
train_split_idx = int(current.shape[0] * 0.70)
val_split_idx = train_split_idx + int(current.shape[0] * 0.10)

train_current = current[:train_split_idx]
val_current = current[train_split_idx:val_split_idx]
test_current = current[val_split_idx:]

train_future_low = future_low[:train_split_idx]
val_future_low = future_low[train_split_idx:val_split_idx]
test_future_low = future_low[val_split_idx:]

train_future_mid = future_mid[:train_split_idx]
val_future_mid = future_mid[train_split_idx:val_split_idx]
test_future_mid = future_mid[val_split_idx:]

train_future_high = future_high[:train_split_idx]
val_future_high = future_high[train_split_idx:val_split_idx]
test_future_high = future_high[val_split_idx:]

print(f"train current: {train_current.shape}")
print(f"val current: {val_current.shape}")
print(f"test current: {test_current.shape}")

print(f"train future low: {train_future_low.shape}")
print(f"val future low: {val_future_low.shape}")
print(f"test future low: {test_future_low.shape}")

print(f"train future mid: {train_future_mid.shape}")
print(f"val future mid: {val_future_mid.shape}")
print(f"test future mid: {test_future_mid.shape}")

print(f"train future high: {train_future_high.shape}")
print(f"val future high: {val_future_high.shape}")
print(f"test future high: {test_future_high.shape}")

train current: (3500, 13)
val current: (500, 13)
test current: (1000, 13)
train future low: (3500, 13)
val future low: (500, 13)
test future low: (1000, 13)
train future mid: (3500, 13)
val future mid: (500, 13)
test future mid: (1000, 13)
train future high: (3500, 13)
val future high: (500, 13)
test future high: (1000, 13)


## Shareability metric

In [200]:
low_shareability, _, _ = ssl_shareability(train_current, train_future_low)
mid_shareability, _, _ = ssl_shareability(train_current, train_future_mid)
high_shareabilitty, _, _ = ssl_shareability(train_current, train_future_high)

print(f"low shareability: {low_shareability}")
print(f"mid shareability: {mid_shareability}")
print(f"high shareability: {high_shareabilitty}")

low shareability: 0.5237084247340593
mid shareability: 0.7611133262980686
high shareability: 0.9992489480932957


## Whiten and center

In [201]:
train_current_whitend_centered = whiten_and_center(train_current)
val_current_whitend_centered = whiten_and_center(val_current)
test_current_whitend_centered = whiten_and_center(test_current)

train_future_low_whitend_centered = whiten_and_center(train_future_low)
val_future_low_whitend_centered = whiten_and_center(val_future_low)
test_future_low_whitend_centered = whiten_and_center(test_future_low)

train_future_mid_whitend_centered = whiten_and_center(train_future_mid)
val_future_mid_whitend_centered = whiten_and_center(val_future_mid)
test_future_mid_whitend_centered = whiten_and_center(test_future_mid)

train_future_high_whitend_centered = whiten_and_center(train_future_high)
val_future_high_whitend_centered = whiten_and_center(val_future_high)
test_future_high_whitend_centered = whiten_and_center(test_future_high)

## Convert to tensros

In [202]:
train_current_tensor = torch.tensor(train_current_whitend_centered, dtype=torch.float32)
val_current_tensor = torch.tensor(val_current_whitend_centered, dtype=torch.float32)
test_current_tensor = torch.tensor(test_current_whitend_centered, dtype=torch.float32)

train_future_low_tensor = torch.tensor(train_future_low_whitend_centered, dtype=torch.float32)
val_future_low_tensor = torch.tensor(val_future_low_whitend_centered, dtype=torch.float32)
test_future_low_tensor = torch.tensor(test_future_low_whitend_centered, dtype=torch.float32)

train_future_mid_tensor = torch.tensor(train_future_mid_whitend_centered, dtype=torch.float32)
val_future_mid_tensor = torch.tensor(val_future_mid_whitend_centered, dtype=torch.float32)
test_future_mid_tensor = torch.tensor(test_future_mid_whitend_centered, dtype=torch.float32)

train_future_high_tensor = torch.tensor(train_future_high_whitend_centered, dtype=torch.float32)
val_future_high_tensor = torch.tensor(val_future_high_whitend_centered, dtype=torch.float32)
test_future_high_tensor = torch.tensor(test_future_high_whitend_centered, dtype=torch.float32)

## Init encoders and optimizers

In [203]:
low_shared_encoder = SharedEncoder(vector_size=13)
low_shared_optimizer = torch.optim.SGD(low_shared_encoder.parameters(), lr=1e-2)

low_seperate_encoder = SeperateEncoder(vector_size=13)
low_seperate_optimizer = torch.optim.SGD(low_seperate_encoder.parameters(), lr=1e-2)

## Low Rank 1 Case

### Shared

In [204]:
epochs = 1000
best_val_loss = float("inf")
delta = 1e-4
best_state = None

for i in range(epochs):
    low_shared_encoder.train()
    Z_x = low_shared_encoder(train_current_tensor)
    Z_y = low_shared_encoder(train_future_low_tensor)
    loss = -torch.abs(torch.mean(Z_x * Z_y))
    train_loss = loss.item()
    low_shared_optimizer.zero_grad()
    loss.backward()
    low_shared_optimizer.step()
    with torch.no_grad():
        weights = low_shared_encoder.shared.weight
        weights.div_(weights.norm(p=2))
             
    low_shared_encoder.eval()
    with torch.no_grad():
        Z_x = low_shared_encoder(val_current_tensor)
        Z_y = low_shared_encoder(val_future_low_tensor)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        val_loss = loss.item()
            
    print(f"epoch: {i+1} train: {train_loss} val: {val_loss}")   
            
    if (val_loss + delta) < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(low_shared_encoder.state_dict())

        
if best_state is not None:
    low_shared_encoder.load_state_dict(best_state)

epoch: 1 train: -0.04362913593649864 val: -0.0874040350317955


epoch: 2 train: -0.10475149750709534 val: -0.08856771886348724
epoch: 3 train: -0.1062912568449974 val: -0.08974545449018478
epoch: 4 train: -0.10784559696912766 val: -0.09093723446130753
epoch: 5 train: -0.10941451042890549 val: -0.09214309602975845
epoch: 6 train: -0.11099794507026672 val: -0.09336304664611816
epoch: 7 train: -0.11259586364030838 val: -0.09459707885980606
epoch: 8 train: -0.11420819163322449 val: -0.09584517031908035
epoch: 9 train: -0.11583490669727325 val: -0.09710736572742462
epoch: 10 train: -0.1174759492278099 val: -0.09838360548019409
epoch: 11 train: -0.11913121491670609 val: -0.09967387467622757
epoch: 12 train: -0.1208006739616394 val: -0.10097818076610565
epoch: 13 train: -0.1224842444062233 val: -0.10229646414518356
epoch: 14 train: -0.12418179214000702 val: -0.1036287471652031
epoch: 15 train: -0.12589330971240997 val: -0.1049749106168747
epoch: 16 train: -0.12761859595775604 val: -0.10633496195077896
epoch: 17 train: -0.12935763597488403 val: -0.10770883

### Seperate

In [205]:
epochs = 1000
best_val_loss = float("inf")
delta = 1e-4
best_state = None

for i in range(epochs):
    low_seperate_encoder.train()
    Z_x, Z_y = low_seperate_encoder(train_current_tensor, train_future_low_tensor)
    loss = -torch.abs(torch.mean(Z_x * Z_y))
    train_loss = loss.item()
    low_seperate_optimizer.zero_grad()
    loss.backward()
    low_seperate_optimizer.step()
    with torch.no_grad():
        current_weights = low_seperate_encoder.current.weight
        future_weights = low_seperate_encoder.future.weight
        
        current_weights.div_(current_weights.norm(p=2)) 
        future_weights.div_(future_weights.norm(p=2))
             
    low_seperate_encoder.eval()
    with torch.no_grad():
        Z_x, Z_y = low_seperate_encoder(val_current_tensor, val_future_low_tensor)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        val_loss = loss.item()
            
    print(f"epoch: {i+1} train: {train_loss} val: {val_loss}")   
            
            
    if (val_loss + delta) < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(low_seperate_encoder.state_dict())

        
if best_state is not None:
    low_seperate_encoder.load_state_dict(best_state)

epoch: 1 train: -0.01858590915799141 val: -0.005673098377883434
epoch: 2 train: -0.06644992530345917 val: -0.006504518445581198
epoch: 3 train: -0.06725571304559708 val: -0.0073462435975670815
epoch: 4 train: -0.06807093322277069 val: -0.008198430761694908
epoch: 5 train: -0.06889577209949493 val: -0.009061257354915142
epoch: 6 train: -0.06973039358854294 val: -0.009934919886291027
epoch: 7 train: -0.07057495415210724 val: -0.010819602757692337
epoch: 8 train: -0.07142964750528336 val: -0.011715492233633995
epoch: 9 train: -0.07229462265968323 val: -0.012622760608792305
epoch: 10 train: -0.07317008078098297 val: -0.013541601598262787
epoch: 11 train: -0.07405617833137512 val: -0.014472189359366894
epoch: 12 train: -0.07495307177305222 val: -0.015414738096296787
epoch: 13 train: -0.07586096972227097 val: -0.016369402408599854
epoch: 14 train: -0.07678002119064331 val: -0.01733640395104885
epoch: 15 train: -0.07771044224500656 val: -0.018315937370061874
epoch: 16 train: -0.07865240424871

### Holdout Set

In [206]:
with torch.no_grad():
    Z_x = low_shared_encoder(test_current_tensor)
    Z_y = low_shared_encoder(test_future_low_tensor)
    low_shared_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"shared loss: {low_shared_loss}")

with torch.no_grad():
    Z_x, Z_y = low_seperate_encoder(test_current_tensor, test_future_low_tensor)
    low_sep_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"seperate loss: {low_sep_loss}")

shared loss: -0.46158891916275024
seperate loss: -0.8853817582130432


### Analysis

In [207]:
low_test_result = low_shared_loss / low_sep_loss
print(f"Low test result: {low_test_result}, Low shareability result: {low_shareability}")

Low test result: 0.5213444828987122, Low shareability result: 0.5237084247340593


## Mid Rank One Case

In [208]:
mid_shared_encoder = SharedEncoder(vector_size=13)
mid_shared_optimizer = torch.optim.SGD(mid_shared_encoder.parameters(), lr=1e-2)

mid_seperate_encoder = SeperateEncoder(vector_size=13)
mid_seperate_optimizer = torch.optim.SGD(mid_seperate_encoder.parameters(), lr=1e-2)

In [209]:
epochs = 1000
best_val_loss = float("inf")
delta = 1e-4
best_state = None

for i in range(epochs):
    train_loss = 0
    mid_shared_encoder.train()
    Z_x = mid_shared_encoder(train_current_tensor)
    Z_y = mid_shared_encoder(train_future_mid_tensor)
    loss = -torch.abs(torch.mean(Z_x * Z_y))
    train_loss += loss.item()
    mid_shared_optimizer.zero_grad()
    loss.backward()
    mid_shared_optimizer.step()
    with torch.no_grad():
        weights = mid_shared_encoder.shared.weight
        weights.div_(weights.norm(p=2))
             
    val_loss = 0
    mid_shared_encoder.eval()
    with torch.no_grad():
        Z_x = mid_shared_encoder(val_current_tensor)
        Z_y = mid_shared_encoder(val_future_mid_tensor)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        val_loss += loss.item()
            
    print(f"epoch: {i+1} train: {train_loss} val: {val_loss}")   
            
    if (val_loss + delta) < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(mid_shared_encoder.state_dict())
    
if best_state is not None:
    mid_shared_encoder.load_state_dict(best_state)
        

epoch: 1 train: -0.004675806500017643 val: -0.020871486514806747
epoch: 2 train: -0.015954837203025818 val: -0.021083220839500427
epoch: 3 train: -0.016198130324482918 val: -0.021294796839356422
epoch: 4 train: -0.01644268073141575 val: -0.021506236866116524
epoch: 5 train: -0.016688555479049683 val: -0.021717611700296402
epoch: 6 train: -0.016935812309384346 val: -0.02192898467183113
epoch: 7 train: -0.017184540629386902 val: -0.022140396758913994
epoch: 8 train: -0.017434805631637573 val: -0.022351916879415512
epoch: 9 train: -0.01768667995929718 val: -0.022563613951206207
epoch: 10 train: -0.01794024370610714 val: -0.02277553267776966
epoch: 11 train: -0.01819556951522827 val: -0.022987743839621544
epoch: 12 train: -0.018452731892466545 val: -0.02320033498108387
epoch: 13 train: -0.018711822107434273 val: -0.023413319140672684
epoch: 14 train: -0.018972912803292274 val: -0.02362680807709694
epoch: 15 train: -0.019236093387007713 val: -0.023840859532356262
epoch: 16 train: -0.0195014

In [210]:
epochs = 1000
best_val_loss = float("inf")
delta = 1e-4
best_state = None

for i in range(epochs):
    mid_seperate_encoder.train()
    Z_x, Z_y = mid_seperate_encoder(train_current_tensor, train_future_mid_tensor)
    loss = -torch.abs(torch.mean(Z_x * Z_y))
    train_loss = loss.item()
    mid_seperate_optimizer.zero_grad()
    loss.backward()
    mid_seperate_optimizer.step()
    with torch.no_grad():
        current_weights = mid_seperate_encoder.current.weight
        future_weights = mid_seperate_encoder.future.weight
        
        current_weights.div_(current_weights.norm(p=2)) 
        future_weights.div_(future_weights.norm(p=2))
             
    mid_seperate_encoder.eval()
    with torch.no_grad():
        Z_x, Z_y = mid_seperate_encoder(val_current_tensor, val_future_mid_tensor)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        val_loss = loss.item()
            
    print(f"epoch: {i+1} train: {train_loss} val: {val_loss}")   
            
    if (val_loss + delta) < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(mid_seperate_encoder.state_dict())
    
if best_state is not None:
    mid_seperate_encoder.load_state_dict(best_state)

epoch: 1 train: -0.00011661843018373474 val: -0.015268288552761078
epoch: 2 train: -0.0005833320901729167 val: -0.015590517781674862


epoch: 3 train: -0.0008775116875767708 val: -0.015908805653452873
epoch: 4 train: -0.0011684374185279012 val: -0.01622321829199791
epoch: 5 train: -0.0014562043361365795 val: -0.0165338646620512
epoch: 6 train: -0.0017408615676686168 val: -0.016840802505612373
epoch: 7 train: -0.0020224947948008776 val: -0.017144130542874336
epoch: 8 train: -0.0023011809680610895 val: -0.017443887889385223
epoch: 9 train: -0.0025769693311303854 val: -0.017740199342370033
epoch: 10 train: -0.0028499357867985964 val: -0.0180331002920866
epoch: 11 train: -0.0031201571691781282 val: -0.018322685733437538
epoch: 12 train: -0.0033876942470669746 val: -0.01860900968313217
epoch: 13 train: -0.0036526056937873363 val: -0.01889217086136341
epoch: 14 train: -0.0039149802178144455 val: -0.019172202795743942
epoch: 15 train: -0.0041748578660190105 val: -0.019449204206466675
epoch: 16 train: -0.00443232012912631 val: -0.019723253324627876
epoch: 17 train: -0.0046874117106199265 val: -0.019994394853711128
epoch: 18 t

In [211]:
with torch.no_grad():
    Z_x = mid_shared_encoder(test_current_tensor)
    Z_y = mid_shared_encoder(test_future_mid_tensor)
    mid_shared_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"shared loss: {mid_shared_loss}")

with torch.no_grad():
    Z_x, Z_y = mid_seperate_encoder(test_current_tensor, test_future_mid_tensor)
    mid_sep_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"seperate loss: {mid_sep_loss}")

shared loss: -0.680409848690033
seperate loss: -0.8850213885307312


In [212]:
mid_test_result = mid_shared_loss / mid_sep_loss
print(f"Mid test result: {mid_test_result}, Mid shareability result: {mid_shareability}")

Mid test result: 0.7688060998916626, Mid shareability result: 0.7611133262980686


## High Rank One Case

In [213]:
high_shared_encoder = SharedEncoder(vector_size=13)
high_shared_optimizer = torch.optim.SGD(high_shared_encoder.parameters(), lr=1e-2)

high_seperate_encoder = SeperateEncoder(vector_size=13)
high_seperate_optimizer = torch.optim.SGD(high_seperate_encoder.parameters(), lr=1e-2)

In [214]:
epochs = 1000
best_val_loss = float("inf")
delta = 1e-4
best_state = None

for i in range(epochs):
    high_shared_encoder.train()
    Z_x = high_shared_encoder(train_current_tensor)
    Z_y = high_shared_encoder(train_future_high_tensor)
    loss = -torch.abs(torch.mean(Z_x * Z_y))
    train_loss = loss.item()
    high_shared_optimizer.zero_grad()
    loss.backward()
    high_shared_optimizer.step()
    with torch.no_grad():
        weights = high_shared_encoder.shared.weight
        weights.div_(weights.norm(p=2))
             
    high_shared_encoder.eval()
    with torch.no_grad():
        Z_x = high_shared_encoder(val_current_tensor)
        Z_y = high_shared_encoder(val_future_high_tensor)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        val_loss = loss.item()
            
    print(f"epoch: {i+1} train: {train_loss} val: {val_loss}")   
            
    if (val_loss + delta) < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(high_shared_encoder.state_dict())

        
    
if best_state is not None:
    high_shared_encoder.load_state_dict(best_state)

epoch: 1 train: -0.006988021079450846 val: -0.01183808222413063
epoch: 2 train: -0.025380097329616547 val: -0.01083116140216589
epoch: 3 train: -0.0265243798494339 val: -0.009789817966520786
epoch: 4 train: -0.02770523726940155 val: -0.00871293991804123
epoch: 5 train: -0.028923779726028442 val: -0.007599370088428259
epoch: 6 train: -0.030181173235177994 val: -0.006447949446737766
epoch: 7 train: -0.03147859871387482 val: -0.005257480777800083
epoch: 8 train: -0.03281725198030472 val: -0.004026759881526232
epoch: 9 train: -0.034198369830846786 val: -0.002754510845988989
epoch: 10 train: -0.03562323376536369 val: -0.0014394550817087293
epoch: 11 train: -0.03709312155842781 val: -8.028793672565371e-05
epoch: 12 train: -0.0386093407869339 val: -0.0013243465218693018
epoch: 13 train: -0.04017326235771179 val: -0.002775777829810977
epoch: 14 train: -0.04178623482584953 val: -0.004275440238416195
epoch: 15 train: -0.04344966262578964 val: -0.005824731662869453
epoch: 16 train: -0.04516497626

In [215]:
epochs = 1000
best_val_loss = float("inf")
delta = 1e-4
best_state = None

for i in range(epochs):
    high_seperate_encoder.train()
    Z_x, Z_y = high_seperate_encoder(train_current_tensor, train_future_high_tensor)
    loss = -torch.abs(torch.mean(Z_x * Z_y))
    train_loss = loss.item()
    high_seperate_optimizer.zero_grad()
    loss.backward()
    high_seperate_optimizer.step()
    with torch.no_grad():
        current_weights = high_seperate_encoder.current.weight
        future_weights = high_seperate_encoder.future.weight
        
        current_weights.div_(current_weights.norm(p=2)) 
        future_weights.div_(future_weights.norm(p=2))
             
    val_loss = 0
    high_seperate_encoder.eval()
    with torch.no_grad():
        Z_x, Z_y = high_seperate_encoder(val_current_tensor, val_future_high_tensor)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        val_loss = loss.item()
            
    print(f"epoch: {i+1} train: {train_loss} val: {val_loss}")   
            
            
    if (val_loss + delta) < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(high_seperate_encoder.state_dict())

    
if best_state is not None:
    high_seperate_encoder.load_state_dict(best_state)

epoch: 1 train: -0.0031117661856114864 val: -0.10006089508533478
epoch: 2 train: -0.008368108421564102 val: -0.09989538043737411
epoch: 3 train: -0.008528312668204308 val: -0.0997292548418045
epoch: 4 train: -0.008688303641974926 val: -0.09956257045269012
epoch: 5 train: -0.008848111145198345 val: -0.09939524531364441
epoch: 6 train: -0.009007764980196953 val: -0.09922730922698975
epoch: 7 train: -0.009167300537228584 val: -0.0990586131811142
epoch: 8 train: -0.009326746687293053 val: -0.09888925403356552
epoch: 9 train: -0.009486128576099873 val: -0.09871914237737656
epoch: 10 train: -0.00964548159390688 val: -0.09854822605848312
epoch: 11 train: -0.009804832749068737 val: -0.09837649762630463
epoch: 12 train: -0.009964210912585258 val: -0.09820394963026047
epoch: 13 train: -0.010123650543391705 val: -0.09803049266338348
epoch: 14 train: -0.010283184237778187 val: -0.09785613417625427
epoch: 15 train: -0.010442830622196198 val: -0.09768083691596985
epoch: 16 train: -0.0106026325374841

In [216]:
with torch.no_grad():
    Z_x = high_shared_encoder(test_current_tensor)
    Z_y = high_shared_encoder(test_future_high_tensor)
    high_shared_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"shared loss: {high_shared_loss}")

with torch.no_grad():
    Z_x, Z_y = high_seperate_encoder(test_current_tensor, test_future_high_tensor)
    high_sep_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"seperate loss: {high_sep_loss}")

shared loss: -0.8887141346931458
seperate loss: -0.8822112083435059


In [217]:
high_test_result = high_shared_loss / high_sep_loss
print(f"High test result: {high_test_result}, High shareability result: {high_shareabilitty}")

High test result: 1.007371187210083, High shareability result: 0.9992489480932957
